# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR<sup>2</sup> dataset ("Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells") using the [`mlcroissant`](https://github.com/mlcommons/croissant) Python library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Get all record sets
record_sets = metadata.recordSets
print(f"Found {len(record_sets)} record sets.\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '-')}")
    # List all fields with their @id
    fields = rs.get('fields', [])
    if fields:
        print("  Fields:")
        for f in fields:
            fname = f.get('name', '-')
            print(f"    @id: {f['@id']}, Name: {fname}, dataType: {f.get('dataType', '-')}")
    print()


## 3. Data Extraction
Load data from a record set into a DataFrame for analysis. 
Below, we extract all records from each record set and preview the data.

You can use the record set and field `@id`s from the overview above.

In [ ]:
# Extract available record set @id's
record_set_ids = [rs['@id'] for rs in metadata.recordSets]
print(f"Record Set IDs: {record_set_ids}")
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"--- Record Set: {record_set_id} ---")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply data processing steps. 

We'll demonstrate filtering, normalization, and grouping on a numeric field from one record set. Please change the `analysis_record_set_id` and field names as needed based on the actual columns loaded above.

In [ ]:
# Choose a record set and numeric field based on the data overview
# You may need to adjust these variable values after running the previous cells

analysis_record_set_id = record_set_ids[0]  # Choose the main protein/quantification table if present
numeric_field_candidates = list(dataframes[analysis_record_set_id].select_dtypes('number').columns)
print(f"Numeric fields (candidates for analysis): {numeric_field_candidates}")

# Choose one numeric field (for demonstration, pick the first)
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    threshold = dataframes[analysis_record_set_id][numeric_field].mean()
    filtered_df = dataframes[analysis_record_set_id][dataframes[analysis_record_set_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field if a string/categorical column exists
    group_field_candidates = list(dataframes[analysis_record_set_id].select_dtypes(include=['object', 'category']).columns)
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[[numeric_field]].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric field available for EDA.")


## 5. Visualization
Visualize the distribution of a numeric field and relationships using simple charts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the main numeric field
if numeric_field_candidates:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[analysis_record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If a second numeric field exists, do a scatter plot
    if len(numeric_field_candidates) > 1:
        second_field = numeric_field_candidates[1]
        plt.figure(figsize=(6, 6))
        plt.scatter(
            dataframes[analysis_record_set_id][numeric_field],
            dataframes[analysis_record_set_id][second_field],
            alpha=0.6
        )
        plt.xlabel(numeric_field)
        plt.ylabel(second_field)
        plt.title(f"Scatter plot of {numeric_field} vs {second_field}")
        plt.show()
else:
    print("No numeric field for visualization.")


## 6. Conclusion
In this notebook, we loaded the FAIR<sup>2</sup> Croissant-structured dataset describing mass spectrometry analysis of extracellular vesicles from human mast cells. 

- We explored available record sets and their fields (by `@id`).
- Loaded data into DataFrames for analysis.
- Performed basic exploratory analysis, including filtering, normalization, grouping, and visualizations.

For detailed data cleaning and domain-specific analysis, consult the field documentation and the original publication/citation. Continue adapting this notebook according to your own investigative needs.
